# Required Skills Of An AI Engineer

Run `4-classify-job-postings.ipynb` first so that `jobs/2-classified-jobs.jsonl` exists.

This notebook uses an LLM to extract the required skills from the AI engineering job postings, saves them to `jobs/3-extracted_skills.jsonl`, and then counts which skills appear most often.

In [1]:
import json
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import Markdown, display

### Load classified jobs from file

In [ ]:
classified_jobs_path = Path("jobs") / "2-classified-jobs.jsonl"
skills_jsonl_path = Path("jobs") / "3-extracted_skills.jsonl"

if not classified_jobs_path.exists():
    raise FileNotFoundError(
        f"Classified jobs JSONL file not found: {classified_jobs_path.resolve()}. Run 4-classify-job-postings.ipynb first."
    )

if classified_jobs_path.stat().st_size == 0:
    raise ValueError(
        "The classified jobs JSONL file is empty. Run 4-classify-job-postings.ipynb first."
    )

jobs_df = pd.read_json(classified_jobs_path, lines=True)

# Keep only the jobs the LLM classified as true AI engineering roles
jobs_df = jobs_df[jobs_df["is_ai_engineering_role"]]

print(f"Loaded {len(jobs_df)} classified AI engineering job postings")

### Define the skill categories

We give the model a small set of categories so that the extracted skills are easier to compare across postings.

In [3]:
skill_categories = [
    "ai-engineering",
    "ml",
    "data",
    "backend",
    "cloud",
    "ops",
    "languages",
    "frontend",
    "other",
]

category_text = "\n".join(f"- {category}" for category in skill_categories)
print(category_text)

- ai-engineering
- ml
- data
- backend
- cloud
- ops
- languages
- frontend
- other


### Define the prompt and output schema

We ask the model to extract concrete skills and return them in a structured format.

In [4]:
client = OpenAI()

instructions = """
You extract required skills from AI engineering job postings.

Return concise normalized skill names like 'python', 'rag', 'sql', 'aws', or 'docker'.
Only include skills that are clearly important for the role.
Assign each skill to one of the provided categories.
""".strip()

skill_schema = {
    "type": "object",
    "properties": {
        "skills": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "skill": {"type": "string"},
                    "category": {"type": "string", "enum": skill_categories},
                },
                "required": ["skill", "category"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["skills"],
    "additionalProperties": False,
}

### Extract the skills for each job posting

We loop over the classified AI engineering jobs and ask the model which skills are required for each role.

In [5]:
skill_results = []

for i, (_, job) in enumerate(jobs_df.iterrows(), start=1):
    title = job["title"]
    description = job["description"]
    job_url = job["job_url"]

    print(f"Extracting skills for job {i}/{len(jobs_df)}: {title}")

    prompt = f"""
Extract the required skills for this AI engineering job posting.

Use only these categories:
{category_text}

Description:
{description}
""".strip()

    response = client.responses.create(
        model="gpt-5.4-mini",
        instructions=instructions,
        input=prompt,
        text={
            "format": {
                "type": "json_schema",
                "name": "ai_engineering_skill_extraction",
                "schema": skill_schema,
                "strict": True,
            },
            "verbosity": "low",
        },
    )

    result = json.loads(response.output_text)

    extracted_skills = [
        {
            "skill": item["skill"].strip().lower(),
            "category": item["category"],
        }
        for item in result["skills"]
    ]

    skill_results.append(
        {
            "title": title,
            "job_url": job_url,
            "extracted_skills": extracted_skills,
            "extracted_skill_labels": [
                f"{item['skill']} [{item['category']}]" for item in extracted_skills
            ],
        }
    )

Extracting skills for job 1/101: AI Software Engineer
Extracting skills for job 2/101: AI Software Engineer-Senior
Extracting skills for job 3/101: Senior AI/ML Engineer - Engineering Excellence (Full Stack Developer) Vice President
Extracting skills for job 4/101: Sr. Gen AI Engineer (USC/GC) W2
Extracting skills for job 5/101: Principal Engineer, Agentic AI
Extracting skills for job 6/101: AI Engineer – Optical Design
Extracting skills for job 7/101: Senior AI Engineer
Extracting skills for job 8/101: Enterprise Automation & AI Engineer
Extracting skills for job 9/101: VP, Agentic AI Engineering
Extracting skills for job 10/101: Senior AI Engineer (USA - Remote)
Extracting skills for job 11/101: Junior AI Engineer
Extracting skills for job 12/101: Founding AI Engineer
Extracting skills for job 13/101: AI Engineer, Voice Systems
Extracting skills for job 14/101: Staff AI Engineer, Voice Agents
Extracting skills for job 15/101: Sr. Managed Services Engineer - AI & Copilot
Extracting sk

### Save the extracted skills

We turn the list of model outputs into a dataframe and save it as a JSONL file for the next steps.

In [6]:
skills_by_job = pd.DataFrame(skill_results)
skills_by_job.to_json(
    skills_jsonl_path, orient="records", lines=True, force_ascii=False
)

print(f"Saved extracted skills to: {skills_jsonl_path.resolve()}")

skills_by_job_to_show = skills_by_job[["title", "extracted_skill_labels"]].copy()
display(skills_by_job_to_show)

Saved extracted skills to: /Users/lukaslechner/PythonProjects/ai-engineering-foundations-labs/1-introduction/jobs/3-extracted_skills.jsonl


,title,extracted_skill_labels
0,AI Software Engineer,"[python [languages], llm integration [ai-engin..."
1,AI Software Engineer-Senior,"[python [languages], llm [ai-engineering], llm..."
2,Senior AI/ML Engineer - Engineering Excellence...,"[python [languages], java [languages], pandas ..."
3,Sr. Gen AI Engineer (USC/GC) W2,"[python [languages], generative ai [ai-enginee..."
4,"Principal Engineer, Agentic AI","[copilot studio [ai-engineering], azure ai fou..."
...,...,...
96,AI Engineer (Agents),"[ai agents [ai-engineering], llm systems [ai-e..."
97,Copilot Developer/AI Engineer,"[microsoft power platform [ai-engineering], co..."
98,Associate AI Engineer,"[python [languages], numpy [data], pandas [dat..."
99,Lead AI Engineer,"[python [languages], sql [languages], flask [b..."


### Count how often each skill appears

Now we switch from per-job extraction to a small data analysis step. We count how many job postings mention each skill.

In [7]:
exploded_skills = skills_by_job.explode("extracted_skills").dropna(
    subset=["extracted_skills"]
)

exploded_skills["skill"] = exploded_skills["extracted_skills"].str["skill"]
exploded_skills["category"] = exploded_skills["extracted_skills"].str["category"]

total_jobs = len(skills_by_job)

category_stats = (
    exploded_skills.groupby("category")
    .size()
    .reset_index(name="count")
    .sort_values(by=["count", "category"], ascending=[False, True])
    .reset_index(drop=True)
)

skill_stats = (
    exploded_skills.groupby(["category", "skill"])
    .size()
    .reset_index(name="count")
    .sort_values(
        by=["count", "category", "skill"],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

category_stats["share_of_jobs"] = category_stats["count"].apply(
    lambda count: f"{count}/{total_jobs} ({round(count / total_jobs * 100, 1)}%)"
)

skill_stats["share_of_jobs"] = skill_stats["count"].apply(
    lambda count: f"{count}/{total_jobs} ({round(count / total_jobs * 100, 1)}%)"
)

### Display the most common skills

We first look at the categories, then at the most common skills overall, and finally at the top skills inside each category.

In [8]:
top_10_skills = skill_stats[["category", "skill", "share_of_jobs"]].head(10).copy()
top_10_skills.columns = ["Category", "Skill", "Share of jobs"]

display(Markdown("### Most common skills overall"))
display(top_10_skills)

for category in category_stats["category"]:
    top_skills = skill_stats[skill_stats["category"] == category][
        ["skill", "share_of_jobs"]
    ].head(10)

    if not top_skills.empty:
        top_skills = top_skills.copy()
        top_skills.columns = ["Skill", "Share of jobs"]
        display(Markdown(f"### Top skills in `{category}`"))
        display(top_skills)

### Most common skills overall

,Category,Skill,Share of jobs
0,languages,python,87/101 (86.1%)
1,ai-engineering,rag,49/101 (48.5%)
2,ai-engineering,prompt engineering,45/101 (44.6%)
3,ai-engineering,llm,43/101 (42.6%)
4,cloud,aws,35/101 (34.7%)
5,ai-engineering,langchain,32/101 (31.7%)
6,cloud,azure,28/101 (27.7%)
7,ops,ci/cd,26/101 (25.7%)
8,ops,docker,26/101 (25.7%)
9,ml,pytorch,24/101 (23.8%)


### Top skills in `ai-engineering`

,Skill,Share of jobs
1,rag,49/101 (48.5%)
2,prompt engineering,45/101 (44.6%)
3,llm,43/101 (42.6%)
5,langchain,32/101 (31.7%)
12,agentic ai,20/101 (19.8%)
16,langgraph,17/101 (16.8%)
20,ai agents,14/101 (13.9%)
21,generative ai,14/101 (13.9%)
28,mcp,12/101 (11.9%)
32,multi-agent systems,10/101 (9.9%)


### Top skills in `ops`

...

,Skill,Share of jobs
0,python,87/101 (86.1%)
18,typescript,16/101 (15.8%)
41,java,9/101 (8.9%)
47,go,8/101 (7.9%)
48,sql,8/101 (7.9%)
57,rust,7/101 (6.9%)
106,c#,4/101 (4.0%)
107,javascript,4/101 (4.0%)
140,scala,3/101 (3.0%)
233,c,2/101 (2.0%)


### Top skills in `other`

,Skill,Share of jobs
157,microsoft 365,3/101 (3.0%)
158,security,3/101 (3.0%)
259,agile,2/101 (2.0%)
260,governance,2/101 (2.0%)
261,responsible ai,2/101 (2.0%)
262,technical leadership,2/101 (2.0%)
263,vscode,2/101 (2.0%)
733,access control,1/101 (1.0%)
734,ai governance,1/101 (1.0%)
735,autodesk,1/101 (1.0%)


### Top skills in `frontend`

,Skill,Share of jobs
68,react,6/101 (5.9%)
105,next.js,4/101 (4.0%)
231,angular,2/101 (2.0%)
232,teams,2/101 (2.0%)
548,bi tools,1/101 (1.0%)
549,data visualization,1/101 (1.0%)
550,desktop applications,1/101 (1.0%)
551,electron,1/101 (1.0%)
552,frontend development,1/101 (1.0%)
553,frontend ui modeling,1/101 (1.0%)
